# Direct regression — zero-inflated drone strike count forecasting

Fair comparison across:
 Gradient boosters (LightGBM / XGBoost / CatBoost) — Poisson and Tweedie tuned **separately**
 Persistence baselines (yesterday, last week same day)
 ARIMA (per-region classical baseline)
 Darts LSTM (BlockRNN)

Every model is fit & evaluated on the **same expanding-window folds** over
the validation segment. Test set is untouched until the final run.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/thesis/CODEBASE'


Mounted at /content/drive


In [3]:
!nvidia-smi

Fri May 22 09:06:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install "darts[all]" statsmodels optuna

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

CUDA available: True
Device: NVIDIA L4


Imports and config

In [6]:
PROJECT_DIR = '/content/drive/MyDrive/thesis/CODEBASE'
%cd $PROJECT_DIR

/content/drive/MyDrive/thesis/CODEBASE


In [8]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import *
from src import _maybe_scale_covs, _skill  # not re-exported by import *

from darts import TimeSeries
from darts.dataprocessing.transformers import (
    WindowTransformer, StaticCovariatesTransformer, Scaler,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
available_threads = get_available_threads()
print(f'CPU count: {available_threads}')

CPU count: 12


In [9]:
# -------------------- CONFIG --------------------
DATA_FOLDER       = "./data"
FIXED_DATA_PATH   = construct_path(DATA_FOLDER, "fixed")
DATASET_PATH      = construct_path(DATA_FOLDER, "dataset")

TARGET            = "act_drone_strike_on_ua"

OUTPUT_CHUNK_LEN  = 7
INPUT_LAGS        = 7
MULTI_MODELS      = True

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20
CV_STRIDE                       = 1

# ---- Model groups ---------------------------------------------------------
NAIVE_MODELS  = {"naive_last", "naive_weekly"}
NEURAL_MODELS  = {"lstm_poisson", "lstm_tweedie"}
LOCAL_MODELS  = {"arima"}

# Each GBM family is split by objective so Optuna tunes Poisson and Tweedie
# in their own studies (see _suggest_*_params + the tuning loop below).
GBM_FAMILIES  = ["lightgbm", "xgboost", "catboost"]
GBM_OBJECTIVES = ["poisson", "tweedie"]
GBM_VARIANTS  = [f"{fam}_{obj}" for fam in GBM_FAMILIES for obj in GBM_OBJECTIVES]
# -> ["lightgbm_poisson", "lightgbm_tweedie",
#     "xgboost_poisson",  "xgboost_tweedie",
#     "catboost_poisson", "catboost_tweedie"]

# Tuned GBMs are added to REGRESSORS_TO_RUN dynamically after Optuna finishes.
REGRESSORS_TO_RUN = [
    "naive_last", "naive_weekly",
    "arima",
]

# Optuna config
OPTUNA_N_TRIALS    = 50   # per study; each Poisson/Tweedie variant gets its own
OPTUNA_TIMEOUT_S   = None  # 1 h cap per variant study

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
import torch
torch.manual_seed(RANDOM_STATE)


## 1. Load data

In [10]:
regions,master_timeseries,regions_activity = load_data(data_path=FIXED_DATA_PATH,dataset_path=DATASET_PATH)

master_timeseries shape: (847, 1622)
regions: 25  -  ['cherkasy', 'chernihiv', 'chernivtsi', 'dnipropetrovsk', 'donetsk'] ...
start: 2022-09-28 00:00:00 end: 2025-01-21 00:00:00


In [11]:
for_global_reset, global_weather_columns = get_engineered_features(
    master_timeseries=master_timeseries,
    data_path=FIXED_DATA_PATH,
    target_col=TARGET,
    regions=regions,
    regions_activity=regions_activity,
    binarize_target=False
)

Shape after feature engineering: (16940, 115)
Target mean: 0.394


In [ ]:
print(for_global_reset.head())

     region event_date  acled_other_rus_armed_clash  acled_other_rus_disrupted_weapons_use  acled_other_rus_shelling_artillery_missile_attack  acled_other_ua_armed_clash  \
0  cherkasy 2022-09-28                          0.0                                    0.0                                                0.0                         0.0   
1  cherkasy 2022-09-29                          0.0                                    0.0                                                0.0                         0.0   
2  cherkasy 2022-09-30                          0.0                                    0.0                                                0.0                         0.0   
3  cherkasy 2022-10-01                          0.0                                    0.0                                                0.0                         0.0   
4  cherkasy 2022-10-02                          0.0                                    0.0                                             

In [ ]:
print(for_global_reset.isna().any()[lambda x: x])
# printfor_global_reset[for_global_reset["Activity_Level"].isna() == True][['Activity_Level','event_date','region']])

Series([], dtype: bool)


In [ ]:
# Future vs past covariate split
holiday_cols, future_covariates, exclude_cols, past_covariates = split_future_and_past_cov(for_global_reset,global_weather_columns,TARGET)

# future covariates : 27
# past covariates   : 85


## Build Darts TimeSeries and apply windowed transforms
Getting lag
 variabels

In [ ]:
target_series_list, past_covs_list,future_covs_list = build_ts_and_apply_window_transformer(for_global_reset,TARGET,past_covariates,future_covariates,ed_alpha=halflife_to_alpha(7))

Past-cov components: raw=85  transformed=595


In [ ]:
raw_past_covs_list = TimeSeries.from_group_dataframe(
    for_global_reset,
    group_cols="region", time_col="event_date",
    value_cols=past_covariates,
)

## Encode static covariates and split 70/10/20


In [ ]:
region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv, TRAIN_VAL_END,CV_START_VAL =\
      get_covs_and_encodings(target_series_list,past_covs_list,future_covs_list,TRAIN_FRAC,VAL_FRAC)

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only


## Hyper-parameter cheat sheet

A compact reference for what to sweep first, second, third.

| Model | First knob (★★★) | Second knob (★★) | Third knob (★) | Notes |
|---|---|---|---|---|
| `lightgbm_*` / `xgboost_*` / `catboost_*` | `n_estimators` + `learning_rate` | `max_depth` / `num_leaves` / `depth` | `min_child_samples` / `min_child_weight` / `l2_leaf_reg` | Poisson and Tweedie are **tuned in separate Optuna studies**. For Tweedie, also sweep `tweedie_variance_power ∈ (1.1, 1.9)`. Poisson vs Tweedie matters most when there is little overdispersion. |
| `poisson_glm` | `alpha` (L2) | `max_iter` | — | Assumes mean == variance. Bad when data is overdispersed. |
| `tweedie_glm` | `power ∈ (1, 2)` | `alpha` (L2) | `max_iter` | `power = 1.5` is a safe starting point for zero-inflated continuous-ish counts. |
| `negbin_glm` | `alpha` (dispersion) | `maxiter` | — | Preferred over Poisson when `Var(Y) > E[Y]`. |
| `lstm` (BlockRNN) | `hidden_dim`, `n_rnn_layers` | `dropout`, `n_epochs` | `batch_size`, `learning_rate` | No count likelihood in BlockRNN — consider scaling the target. |
| `arima` | `p`, `q` | `d` | — | Local per region; consider `AutoARIMA` once for each region to find starting orders. |
| `naive_*` | — | — | — | No parameters. These are your floor. |

**Sanity rule**: whichever model you pick, its global `MASE_mean` must be
below 1 (i.e. `SkillRMSE` > 0 vs `naive_weekly`). If it isn't, the
covariates aren't carrying signal — inspect per-horizon metrics (it's
common for h=1 to be easy and h=7 to collapse).


## Regressors

Every tabular model shares the same `lags`, `output_chunk_length`,
`multi_models` and `add_encoders` setup so the comparison is fair —
**only the learner and its loss/likelihood change**. The LSTM uses the
same input / output chunk lengths; ARIMA is local by construction.

**What's worth tuning** (marked `★` = high impact, `·` = secondary):
- **Gradient boosters**: `★ n_estimators + learning_rate` (trade-off), `★ max_depth / num_leaves`,
  `★ min_child_samples` (zero-inflation: too-small splits overfit zeros), `· subsample`,
  `· colsample_bytree`, `· reg_alpha / reg_lambda`. For Tweedie, also `★ tweedie_variance_power` ∈ (1, 2).
  Poisson and Tweedie are tuned in **separate studies** so each objective gets a
  fair search budget.
- **GLMs**: `★ alpha` (L2), `★ power` for Tweedie, `★ dispersion (alpha)` for NegBin,
  `· max_iter`. Standardise inputs — these are linear in the features.
- **LSTM (BlockRNN)**: `★ hidden_dim`, `★ n_rnn_layers`, `★ dropout`, `★ n_epochs`,
  `· batch_size`, `· learning_rate`. Consider target scaling since there's no count likelihood.
- **ARIMA**: `★ p, q` (grid `p ∈ {1,3,7,14}`, `q ∈ {0,1,2}`), `★ d ∈ {0, 1}`.
  Use `AutoARIMA` if you want automatic order selection (slower).
- **Naive**: no parameters — they are your floor. Any model must beat both.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# https://github.com/sktime/pytorch-forecasting/blob/main/pytorch_forecasting/metrics/point.py
class PoissonNLLLogLink(nn.Module):
    def forward(self, y_pred, y_true):
        return F.poisson_nll_loss(
            y_pred, y_true, log_input=True, full=False, reduction="mean" # Darts expects a scalar loss back so reduction = mean
        )


class TweedieNLLLogLink(nn.Module):
    def __init__(self, power: float = 1.5):
        super().__init__()
        if not (1.0 < power < 2.0):
            raise ValueError("power must lie strictly in (1, 2)")
        self.power = power

    def forward(self, y_pred, y_true):
        p  = self.power
        mu = torch.exp(y_pred)
        a  = -y_true * torch.pow(mu, 1.0 - p) / (1.0 - p)
        b  =           torch.pow(mu, 2.0 - p) / (2.0 - p)
        return torch.mean(a + b)

def build_lstm_count(objective_kind: str, params: dict, *, tweedie_power: float = 1.5):
    """LSTM in deterministic point-forecast mode, trained with Poisson or
    Tweedie NLL on a log-link. Mirrors GBDT objectives for fair comparison.

    The network parameterizes log(μ); predictions come out in log-space and
    are inverted via exp() inside run_expanding_cv (gated on _count_log_link).
    """
    if objective_kind == "poisson":
        loss_fn = PoissonNLLLogLink()
    elif objective_kind == "tweedie":
        loss_fn = TweedieNLLLogLink(power=tweedie_power)
    else:
        raise ValueError(f"unknown objective_kind {objective_kind!r}")

    p = dict(params)
    p.pop("likelihood", None)        # force deterministic — no sampling
    p["loss_fn"]    = loss_fn
    p["likelihood"] = None

    from darts.models import BlockRNNModel
    m = BlockRNNModel(**p)
    m._count_log_link = True         # signal to run_expanding_cv to invert the link
    return m

In [ ]:
from pytorch_lightning.callbacks import EarlyStopping
# --- Shared forecasting skeleton: every tabular model gets the same inputs ---
COMMON_KWARGS_TAB = get_common_kwargs()

ES_NN  = EarlyStopping(monitor="val_loss",   patience=10, min_delta=1e-4, mode="min")
# ES = EarlyStopping(
#     monitor="train_loss",   # use "val_loss" if you pass val_series to .fit()
#     patience=5,
#     min_delta=1e-4,
#     mode="min",
# )

NN_TRAINER_KWARGS = dict(
    accelerator          = "gpu",
    devices = 1,
    precision = "32-true",
    enable_progress_bar  = True,
    enable_model_summary = True,
    log_every_n_steps    = 10,
    callbacks            = [ES_NN],
    gradient_clip_val    = 1.0,
)


# ---------------------- build_regressor ----------------------
def build_regressor(name: str):
    """Return a Darts forecasting model ready to ``.fit()``.

    Every branch uses the same ``INPUT_LAGS`` / ``OUTPUT_CHUNK_LEN`` so that
    a 7-day forecast from yesterday is directly comparable across models.
    These default-config branches are used for the feature-selection pass and
    as a fallback; the Optuna-tuned versions go through ``build_gbm_from_params``.
    """
    name = name.lower()

    # ======================================================================
    # 1) Gradient boosters — Darts native wrappers (default configs)
    # ======================================================================
    if name.startswith("lightgbm_poisson"):
        from darts.models import LightGBMModel
        return LightGBMModel(
            **COMMON_KWARGS_TAB,
            objective         = "poisson",
            # --- tuning surface ---
            num_leaves        = 31,      # ★ 15–127 (capacity)
            max_depth         = 5,       # ★ -1 or 3–10
            min_child_samples = 30,      # ★ 10–200 — zero-heavy targets need higher
            subsample         = 0.8,     # · 0.6–1.0
            colsample_bytree  = 0.8,     # · 0.6–1.0
            learning_rate     = 0.05,    # ★ pair with n_estimators
            n_estimators      = 500,     # ★ use early stopping on a held-out val
            reg_alpha         = 0.0,     # · L1
            reg_lambda        = 0.0,     # · L2
            random_state      = RANDOM_STATE,
            verbose           = -1,
            device_type  = "gpu",
            # device_type       = "cpu",
            # num_threads = available_threads,
            force_col_wise = True,
        )
    if name.startswith("lightgbm_tweedie"):
        from darts.models import LightGBMModel
        return LightGBMModel(
            **COMMON_KWARGS_TAB,
            objective              = "tweedie",
            # --- tuning surface ---
            num_leaves             = 31,      # ★ 15–127 (capacity)
            max_depth              = 5,       # ★ -1 or 3–10
            min_child_samples      = 30,      # ★ 10–200 — zero-heavy targets need higher
            subsample              = 0.8,     # · 0.6–1.0
            colsample_bytree       = 0.8,     # · 0.6–1.0
            learning_rate          = 0.05,    # ★ pair with n_estimators
            n_estimators           = 500,     # ★ use early stopping on a held-out val
            reg_alpha              = 0.0,     # · L1
            reg_lambda             = 0.0,     # · L2
            tweedie_variance_power = 1.5,
            random_state           = RANDOM_STATE,
            verbose                = -1,
            device_type  = "gpu",
            # device_type            = "cpu",
            # num_threads = available_threads,
            force_col_wise = True,
        )

    if name.startswith("xgboost_poisson"):
        from darts.models import XGBModel
        return XGBModel(
            **COMMON_KWARGS_TAB,
            objective         = "count:poisson",
            # --- tuning surface ---
            max_depth         = 5,       # ★ 3–10
            min_child_weight  = 3,       # ★ 1–10
            subsample         = 0.8,     # · 0.6–1.0
            colsample_bytree  = 0.8,     # · 0.6–1.0
            learning_rate     = 0.05,    # ★
            n_estimators      = 500,     # ★
            reg_alpha         = 0.0,     # · L1
            reg_lambda        = 1.0,     # · L2
            tree_method       = "hist",
            device       = "cuda",
            # device            = "cpu",
            # n_jobs = available_threads,
            random_state      = RANDOM_STATE,
            verbosity         = 0,
        )
    if name.startswith("xgboost_tweedie"):
        from darts.models import XGBModel
        return XGBModel(
            **COMMON_KWARGS_TAB,
            objective              = "reg:tweedie",
            # --- tuning surface ---
            max_depth              = 5,       # ★ 3–10
            min_child_weight       = 3,       # ★ 1–10
            subsample              = 0.8,     # · 0.6–1.0
            colsample_bytree       = 0.8,     # · 0.6–1.0
            learning_rate          = 0.05,    # ★
            n_estimators           = 500,     # ★
            reg_alpha              = 0.0,     # · L1
            reg_lambda             = 1.0,     # · L2
            tweedie_variance_power = 1.5,
            tree_method            = "hist",
            device       = "cuda",
            # device                 = "cpu",
            # n_jobs = available_threads,
            random_state           = RANDOM_STATE,
            verbosity              = 0,
        )

    if name.startswith("catboost_poisson"):
        from darts.models import CatBoostModel
        return CatBoostModel(
            **COMMON_KWARGS_TAB,
            loss_function     = "Poisson",
            boost_from_average= False,
            # --- tuning surface ---
            depth             = 5,       # ★ 4–10
            learning_rate     = 0.05,    # ★
            iterations        = 500,     # ★ equivalent of n_estimators
            l2_leaf_reg       = 3,       # ★ 1–10
            subsample         = 0.8,     # · (bootstrap_type=Bernoulli)
            bootstrap_type    = "Bernoulli",
            task_type         = "GPU",
            # task_type         = "CPU",
            # thread_count = available_threads,
            random_seed       = RANDOM_STATE,
            verbose           = False,
        )
    if name.startswith("catboost_tweedie"):
        from darts.models import CatBoostModel
        return CatBoostModel(
            **COMMON_KWARGS_TAB,
            loss_function     = "Tweedie:variance_power=1.5",
            boost_from_average= False,
            # --- tuning surface ---
            depth             = 5,       # ★ 4–10
            learning_rate     = 0.05,    # ★
            iterations        = 500,     # ★ equivalent of n_estimators
            l2_leaf_reg       = 3,       # ★ 1–10
            subsample         = 0.8,     # · (bootstrap_type=Bernoulli)
            bootstrap_type    = "Bernoulli",
            task_type         = "GPU",
            # task_type         = "CPU",
            # thread_count = available_threads,
            random_seed       = RANDOM_STATE,
            verbose           = False,
        )

    if name == "lstm":
        from darts.models import BlockRNNModel
        return BlockRNNModel(
            model                = "LSTM",
            input_chunk_length   = INPUT_LAGS,
            output_chunk_length  = OUTPUT_CHUNK_LEN,
            hidden_dim           = 32,       # ★ 16–128
            n_rnn_layers         = 1,        # ★ 1–3
            dropout              = 0.1,      # ★ 0.0–0.3
            batch_size           = 64,       # · 32–256
            n_epochs             = 30,       # ★ add EarlyStopping
            random_state         = RANDOM_STATE,
            add_encoders = {"cyclic": {"future": ["month", "week", "dayofweek"]}},
            pl_trainer_kwargs    = NN_TRAINER_KWARGS,
        )

    # LOCAL model (one fit per region, no covariates here)
    if name == "arima":
        from darts.models import ARIMA
        return ARIMA(
            p = 7,              # ★ AR order: try {1, 3, 7, 14}
            d = 0,              # ★ diff order: 0 (stationary-ish) or 1
            q = 1,              # ★ MA order: try {0, 1, 2}
            random_state = RANDOM_STATE,
        )

    if name in NAIVE_MODELS:
        return None

    raise ValueError(f"Unknown regressor name: {name!r}")


# ---------------------------------------------------------------------------
# Param-driven GBM builder used by Optuna trials.
# ``variant`` is a string like "lightgbm_poisson" / "xgboost_tweedie" — the
# objective is fixed by the variant, NOT a tunable parameter.
# Same COMMON_KWARGS_TAB so models stay comparable across the lineup.
# ---------------------------------------------------------------------------
def build_gbm_from_params(variant: str, params: dict):
    variant = variant.lower()
    family, objective_kind = variant.split("_", 1)
    if objective_kind not in {"poisson", "tweedie"}:
        raise ValueError(f"Unknown objective kind in variant {variant!r}")
    p = dict(params)  # copy so we can pop

    if family == "lightgbm":
        from darts.models import LightGBMModel
        if objective_kind == "tweedie":
            extra = {"objective": "tweedie",
                     "tweedie_variance_power": p.pop("tweedie_variance_power")}
        else:
            extra = {"objective": "poisson"}
        return LightGBMModel(
            **COMMON_KWARGS_TAB,
            random_state = RANDOM_STATE,
            verbose      = -1,
            # device_type  = "gpu",
            device_type            = "cpu",
            num_threads = available_threads,
            force_col_wise = True,
            **p, **extra,
        )

    if family == "xgboost":
        from darts.models import XGBModel
        if objective_kind == "tweedie":
            extra = {"objective": "reg:tweedie",
                     "tweedie_variance_power": p.pop("tweedie_variance_power")}
        else:
            extra = {"objective": "count:poisson"}
        return XGBModel(
            **COMMON_KWARGS_TAB,
            tree_method  = "hist",
            # device       = "cuda",
            device                 = "cpu",
            n_jobs = available_threads,
            random_state = RANDOM_STATE,
            verbosity    = 0,
            **p, **extra,
        )

    if family == "catboost":
        from darts.models import CatBoostModel
        if objective_kind == "tweedie":
            vp = p.pop("tweedie_variance_power")
            loss = f"Tweedie:variance_power={vp}"
        else:
            loss = "Poisson"
        return CatBoostModel(
            **COMMON_KWARGS_TAB,
            loss_function      = loss,
            boost_from_average = False,
            bootstrap_type     = "Bernoulli",
            # task_type          = "GPU",
            task_type         = "CPU",
            thread_count = available_threads,
            random_seed        = RANDOM_STATE,
            verbose            = False,
            **p,
        )

    raise ValueError(f"Unknown GBM family: {family!r}")

def build_nn_from_params(family: str, params: dict):
    """LSTM only """
    p = dict(params)
    if family == "lstm":
        from darts.models import BlockRNNModel
        return BlockRNNModel(**p)
    raise ValueError(f"Unknown NN family: {family!r}")


# Build simple LGBM For feature selection

In [ ]:
# A single LightGBM-Tweedie model, trained on everything, purely to rank features
import pickle
from pathlib import Path
path = Path('./features/countreg_saved_sets.pkl')
if path.exists():
    with open(path, "rb") as f:
        region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv, TRAIN_VAL_END, CV_START_VAL = pickle.load(f)
    print(f"Found saved sets in {path}")
else:
    model_feature_selection = build_regressor("lightgbm_tweedie")
    model_feature_selection.fit(
        series            = train_target,
        past_covariates   = full_past_covs,
        future_covariates = full_fut_covs,
    )
    gain_imps = {"Feature": model_feature_selection.lagged_feature_names}

    underlying_model = model_feature_selection.model

    if hasattr(underlying_model, "estimators_"):
        estimators = underlying_model.estimators_
    else:
        estimators = [underlying_model]

    for h, est in enumerate(estimators, start=1):
        gain_imps[f"h{h}_gain"] = est.booster_.feature_importance(importance_type='gain')

    df_gain = pd.DataFrame(gain_imps)
    df_gain["agg_gain"] = df_gain.iloc[:, 1:].mean(axis=1)

    df_gain = df_gain.sort_values(by="agg_gain", ascending=False).reset_index(drop=True)

    df_gain.to_csv('prelimFeatureImportanceRegressor.csv', index=False)
    threshold = .99
    print(df_gain.head(20))
    top_100_features = df_gain.head(100)['Feature'].to_list()
    top_100_features_dict = clean_feature_names(top_100_features)

    past_covs_list   = [subset_safe(ts, top_100_features_dict['pastcov_features_base']) for ts in past_covs_list]
    future_covs_list = [subset_safe(ts, top_100_features_dict['futcov_features_base'])  for ts in future_covs_list]

    region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv, TRAIN_VAL_END, CV_START_VAL = \
        get_covs_and_encodings(target_series_list, past_covs_list, future_covs_list, TRAIN_FRAC, VAL_FRAC)

    sample = past_covs_list[0]
    base_set = top_100_features_dict['pastcov_features_base']
    present  = base_set & set(sample.components)
    missing  = base_set - set(sample.components)
    print(f"{len(present)}/{len(base_set)} base features present in past_covs")
    if missing:
        print("missing (first 10):", list(missing)[:10])
    sample = future_covs_list[0]
    base_set = top_100_features_dict['futcov_features_base']
    present  = base_set & set(sample.components)
    missing  = base_set - set(sample.components)
    print(f"{len(present)}/{len(base_set)} base features present in future_covs")
    if missing:
        print("missing (first 10):", list(missing)[:10])


    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "wb") as f:
        pickle.dump((region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv, TRAIN_VAL_END, CV_START_VAL), f)
    print(f"Saved computed sets to {path}")

Found saved sets in features/countreg_saved_sets.pkl


In [ ]:
_, _, _, _, full_raw_past_covs_LSTM, _, _, _, _ = \
    get_covs_and_encodings(target_series_list, raw_past_covs_list, future_covs_list, TRAIN_FRAC, VAL_FRAC)

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only


##  Evaluation utilities

Metric suite chosen for **zero-inflated count-series** forecasting. Each
metric earns its place by answering a question the others can't:

| Metric | Answers | Why it fits zero-inflated counts |
|---|---|---|
| **MAE** | Average miss size | Robust to the many zeros (L1 loss, no squared explosion) |
| **RMSE** | Worst-miss sensitivity | Penalises the few days where we badly under-forecast a big strike wave |
| **MedAE** | Typical-day miss | Ignores the tail; useful when the long right-tail dominates RMSE |
| **ME** | Systematic bias | `+` = over-forecasting strikes, `-` = under-forecasting. Naive-model ME should be near 0 |
| **PoissonDev** | Count-likelihood fit | Proper scoring rule for counts; natural loss for the Poisson regressors |
| **TweedieDev** (power=1.5) | Zero-inflated fit | Proper scoring rule that rewards predicting exact zeros correctly |
| **ZeroAcc** | Zero-prediction sanity | Fraction of days where `pred < 0.5` matches `y_true == 0` |
| **MASE** | Did we beat seasonal-naive? | Scale-free; divides MAE by in-sample MAE of a weekly naive. `< 1` = better than naive |
| **RMSSE** | Squared-error variant of MASE | The M5-competition standard |


Two views of quality:

| scope              | granularity                           |
| ------------------ | ------------------------------------- |
| `per_region`       | one row per region (pooled folds + horizons) |
| `per_horizon`      | one row per horizon step 1..7 |
| `per_region_horizon` | region × horizon — feeds the heatmap  |
| `global`           | pooled metrics + MASE/RMSSE averaged across regions |


In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    mean_poisson_deviance, mean_tweedie_deviance,
)

EPS               = 1e-9   # guards log / ratio when preds land at zero
NAIVE_SEASONALITY = 7      # weekly seasonality for MASE / RMSSE scales

# ----------------------------------------------------------------------
# Per-region in-sample scales for MASE / RMSSE
# ----------------------------------------------------------------------
# MASE (Hyndman-Koehler 2006) is the gold standard scale-free metric for
# forecast evaluation. It divides the out-of-sample MAE by the MAE of a
# seasonal-naive forecast on the *training* data. Result < 1 => beats the
# in-sample seasonal-naive benchmark. Because it normalises per series it
# is the only sensible way to average errors across regions of very
# different activity levels (donetsk vs chernivtsi etc.).
# RMSSE (from the M5 competition) is its squared-error twin.
# compute_naive_scales imported from src

# evaluate_long imported from src

MAE_SCALES, RMSE_SCALES = compute_naive_scales(
    train_target, region_names, seasonality=NAIVE_SEASONALITY,
)


In [ ]:
# plot_region_horizon_heatmap imported from src



## Naive baselines

Two rock-bottom floors, both produced as Darts `TimeSeries` so they feed the
exact same evaluator as every other model.

- **`naive_last`** — persistence: `y_hat(t+h) = y(t-1)` for every horizon step.
  (*'today's prediction = yesterday's value, repeated'*)
- **`naive_weekly`** — seasonal naive: `y_hat(t+h) = y(t+h-7)`.
  (*'forecast this week = last week, shifted forward'*)

Any serious model must beat both. Weekly naive is especially hard to beat on
calendar-driven targets.


In [ ]:
# naive_last_historical_forecasts imported from src



In [ ]:
# naive_collect_long imported from src



## Train + cross-validate every model

Each non-naive model is trained with an **expanding-window backtest on the
validation segment only**. Folds are weekly (stride = `CV_STRIDE`) so each
7-day prediction window is non-overlapping. The test set (last 20 %) is
never touched here — it only sees the winning configuration once, at the end.

The loop dispatches by model family:
- **Global tabular / neural** (`GBM_MODELS`, `GLM_MODELS`, `NEURAL_MODELS`) —
  one Darts model fit across all regions, using past + future covariates.
  Neural models get their covariates scaled (targets are left on the original
  scale to keep count-style losses meaningful).
- **Local** (`LOCAL_MODELS`, currently just ARIMA) — a fresh model per region,
  no covariates.
- **Naive** — closed-form shift operations, no fitting at all.


In [ ]:
# _maybe_scale_covs imported from src

def run_manual_expanding_cv(name):
    """Expanding-window CV with decoupled predict / retrain strides.

    Predicts every CV_STRIDE days; retrains every OUTPUT_CHUNK_LEN days.
    Between retrains the frozen model predicts from a growing context window.
    Returns ``list[list[TimeSeries]]`` — outer by region, inner by prediction step.
    """
    ref_ts    = target_for_cv[0]                 # shared time axis
    n_total   = len(ref_ts)
    start_idx = int(CV_START_VAL * n_total)

    n_regions      = len(target_for_cv)
    all_fold_preds = [[] for _ in range(n_regions)]
    n_preds    = 0
    n_retrains = 0
    model      = None
    _local_builder = None

    is_local   = name in LOCAL_MODELS
    is_neural  = name in NEURAL_MODELS
    needs_scaling = is_neural

    past_for_fit, fut_for_fit = _maybe_scale_covs(
        full_past_covs, full_fut_covs, do_scale=needs_scaling,
    )

    for t0 in range(start_idx, n_total - OUTPUT_CHUNK_LEN + 1, CV_STRIDE):
        steps_since_start = t0 - start_idx
        split_time        = ref_ts.time_index[t0]

        if steps_since_start % OUTPUT_CHUNK_LEN == 0:
            train_series = [ts.drop_after(split_time) for ts in target_for_cv]
            if is_local:
                _local_builder = lambda n=name: build_regressor(n)
            else:
                model = build_regressor(name)
                fit_kwargs = {"series": train_series}
                if model.supports_past_covariates:
                    fit_kwargs["past_covariates"] = past_for_fit
                if model.supports_future_covariates:
                    fit_kwargs["future_covariates"] = fut_for_fit
                model.fit(**fit_kwargs)
            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {split_time.date()})")

        pred_series = [ts.drop_after(split_time) for ts in target_for_cv]

        if is_local:
            # One model per region; ARIMA doesn't use covariates here.
            preds = []
            for ts in pred_series:
                m = _local_builder()
                m.fit(ts)
                preds.append(m.predict(n=OUTPUT_CHUNK_LEN))
        else:
            pred_kwargs = {"n": OUTPUT_CHUNK_LEN, "series": pred_series}
            if model.supports_past_covariates:
                pred_kwargs["past_covariates"] = past_for_fit
            if model.supports_future_covariates:
                pred_kwargs["future_covariates"] = fut_for_fit

            # Probabilistic models (e.g. TFT + QuantileRegression) need
            # multiple MC samples before we can take a point estimate.
            is_probabilistic = is_neural and getattr(model, "likelihood", None) is not None
            if is_probabilistic:
              pred_kwargs["num_samples"] = 200
            preds = model.predict(show_warnings=False, **pred_kwargs)
            if is_probabilistic:
              preds = [p.quantile(0.5) for p in preds]  # median = point forecast

        for r_idx, pred in enumerate(preds):
            all_fold_preds[r_idx].append(pred)

        n_preds += 1

    print(f"   {n_preds} predictions, {n_retrains} retrains complete")
    return all_fold_preds

In [ ]:
# run_expanding_cv_iter imported from src



In [ ]:
from pytorch_lightning.callbacks import EarlyStopping
import optuna

# ---------------------------------------------------------------------------
# Generalized expanding-window CV: works for the val phase (target_for_cv,
# CV_START_VAL) and for the test phase (full series, TRAIN_VAL_END).
# Returns fold predictions in the same shape as run_manual_expanding_cv.
# ---------------------------------------------------------------------------
# run_expanding_cv imported from src

def _suggest_lightgbm_params(trial, objective_kind: str):
    p = {
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1000, step=100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    if objective_kind == "tweedie":
        p["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)
    return p


def _suggest_xgboost_params(trial, objective_kind: str):
    p = {
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000, step=100),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    if objective_kind == "tweedie":
        p["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)
    return p


def _suggest_catboost_params(trial, objective_kind: str):
    p = {
        "depth":         trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "iterations":    trial.suggest_int("iterations", 200, 1000, step=100),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1.0, 5.0, step=0.5),
        "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
    }
    if objective_kind == "tweedie":
        p["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)
    return p


SUGGESTERS_BY_FAMILY = {
    "lightgbm": _suggest_lightgbm_params,
    "xgboost":  _suggest_xgboost_params,
    "catboost": _suggest_catboost_params,
}


def _nn_trainer_kwargs(trial=None):
    """Trainer kwargs with early stopping. Optuna trial enables LightningPruning."""
    callbacks = [EarlyStopping(monitor="train_loss", patience=5, min_delta=1e-4, mode="min")]

    if trial is not None:
        try:
            from optuna.integration import PyTorchLightningPruningCallback
            callbacks.append(PyTorchLightningPruningCallback(trial, monitor="train_loss"))
        except ImportError:
            pass

    return dict(
        accelerator          = "auto",
        enable_progress_bar  = False,
        enable_model_summary = False,
        log_every_n_steps    = 10,
        gradient_clip_val    = 1.0,
        callbacks            = callbacks,
        logger               = False,
    )


# TFT removed from comparison — only LSTM remains in the NN search space.
def _suggest_lstm_params(trial, objective_kind: str):
    fc_choice = trial.suggest_categorical("hidden_fc_sizes", ["none", "32", "64", "64_32"])
    fc_map = {"none": [], "32": [32], "64": [64], "64_32": [64, 32]}

    params = dict(
        model               = trial.suggest_categorical("model", ["LSTM", "GRU"]),
        input_chunk_length  = trial.suggest_categorical("input_chunk_length", [7, 14, 21, 28]),
        output_chunk_length = OUTPUT_CHUNK_LEN,
        hidden_dim          = trial.suggest_categorical("hidden_dim", [16, 32, 64, 128]),
        n_rnn_layers        = trial.suggest_int("n_rnn_layers", 1, 3),
        hidden_fc_sizes     = fc_map[fc_choice],
        dropout             = trial.suggest_float("dropout", 0.0, 0.4),
        batch_size          = trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        n_epochs            = 100,
        optimizer_kwargs    = {
            "lr":           trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        },
        random_state        = RANDOM_STATE,
        add_encoders        =                 {"cyclic": {
            "past": ["month", "week", "dayofyear", "dayofweek", "day"]
            }},
        pl_trainer_kwargs   = _nn_trainer_kwargs(trial),
    )
    extras = {}
    if objective_kind == "tweedie":
        extras["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.9)
    return params, extras

def _score_fold_preds(fold_preds, target_list, region_names_, metric="MASE_mean"):
    """Run our existing metric pipeline on a fold-pred bundle. Returns the chosen global metric."""
    long_df = collect_predictions_long(target_list, fold_preds, region_names_)
    res     = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES)
    return float(res["global"][metric])


def make_gbm_objective(variant: str):
    """Optuna objective that reports per-fold cumulative score so
    MedianPruner can actually fire."""
    family, objective_kind = variant.lower().split("_", 1)
    suggester = SUGGESTERS_BY_FAMILY[family]

    def _objective(trial):
        params  = suggester(trial, objective_kind)
        builder = lambda: build_gbm_from_params(variant, params)

        last_score = None
        for step, cumulative_fold_preds in enumerate(run_expanding_cv_iter(
            builder,
            target_list    = target_for_cv,
            start_frac     = CV_START_VAL,
            horizon        = OUTPUT_CHUNK_LEN,
            stride         = CV_STRIDE,
            retrain_stride = OUTPUT_CHUNK_LEN,
            past_covs      = full_past_covs,
            future_covs    = full_fut_covs,
            verbose        = False,
        )):
            last_score = _score_fold_preds(
                cumulative_fold_preds, target_for_cv, region_names,
                metric="RMSSE_mean",
            )
            trial.report(last_score, step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(last_score)

    return _objective


def make_nn_objective(variant: str):
    """One Optuna study per LSTM variant: 'lstm_poisson' or 'lstm_tweedie'."""
    family, objective_kind = variant.lower().split("_", 1)
    if family != "lstm" or objective_kind not in {"poisson", "tweedie"}:
        raise ValueError(f"unsupported NN variant {variant!r}")

    def _objective(trial):
        params, extras = _suggest_lstm_params(trial, objective_kind)
        tw_p = extras.get("tweedie_variance_power", 1.5)
        builder = lambda: build_lstm_count(objective_kind, params, tweedie_power=tw_p)

        fold_preds = run_expanding_cv(
            builder,
            target_list    = target_for_cv,
            start_frac     = CV_START_VAL,
            horizon        = OUTPUT_CHUNK_LEN,
            stride         = CV_STRIDE,
            retrain_stride = OUTPUT_CHUNK_LEN,
            past_covs      = full_raw_past_covs_LSTM,
            future_covs    = full_fut_covs,
            is_neural      = True,
            verbose        = False,
        )
        return _score_fold_preds(
            fold_preds, target_for_cv, region_names, metric="RMSSE_mean"
        )

    return _objective

In [ ]:
from pathlib import Path
import pickle

optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_CKPT = Path(PROJECT_DIR) / "checkpoints_tune"
TUNE_CKPT.mkdir(exist_ok=True)

best_params_by_variant = {}
studies                = {}
NN_VARIANTS = ["lstm_poisson", "lstm_tweedie"]
all_variants = GBM_VARIANTS    # 6 GBM + 2 LSTM
# all_variants = NN_VARIANTS    # 6 GBM + 2 LSTM

for variant in all_variants:
    ckpt = TUNE_CKPT / f"{variant}_best.pkl"
    if ckpt.exists():
        with open(ckpt, "rb") as f:
            best_params_by_variant[variant], studies[variant] = pickle.load(f)
        print(f"[{variant}] loaded cached best_value={studies[variant].best_value:.4f}")
        continue

    is_nn = variant in NN_VARIANTS
    n_trials = 25 if is_nn else OPTUNA_N_TRIALS
    print(f"\n=== Tuning {variant} ({n_trials} trials) ===")

    study = optuna.create_study(
        direction = "minimize",
        sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner    = optuna.pruners.MedianPruner(n_warmup_steps=5),
    )

    if is_nn:
        study.optimize(
            make_nn_objective(variant),
            n_trials = n_trials,
            timeout  = OPTUNA_TIMEOUT_S if "OPTUNA_TIMEOUT_S" in dir() else None,
        )
    else:
        study.optimize(
            make_gbm_objective(variant),
            n_trials = n_trials,
            timeout  = OPTUNA_TIMEOUT_S if "OPTUNA_TIMEOUT_S" in dir() else None,
            show_progress_bar = True,
            n_jobs = 1
        )

    best_params_by_variant[variant] = study.best_params
    studies[variant]                = study

    with open(ckpt, "wb") as f:
        pickle.dump((study.best_params, study), f)

    print(f"[{variant}] best score (val CV) = {study.best_value:.4f}")
    print(f"[{variant}] best params: {study.best_params}")

In [ ]:
import pickle
from pathlib import Path

CKPT = Path(PROJECT_DIR) / "checkpoints"
CKPT.mkdir(exist_ok=True)

long_by_model       = {}
fold_preds_by_model = {}


def _build_lstm_from_best(variant: str, best: dict):
    """Reconstruct the LSTM trial params from study.best_params for a re-run.
    `study.best_params` is flat; rebuild the same dict shape _suggest_lstm_params
    produces, then strip tweedie_variance_power into a separate kwarg."""
    fc_map = {"none": [], "32": [32], "64": [64], "64_32": [64, 32]}
    b      = dict(best)
    tw_p   = b.pop("tweedie_variance_power", 1.5)

    params = dict(
        model               = b["model"],
        input_chunk_length  = b["input_chunk_length"],
        output_chunk_length = OUTPUT_CHUNK_LEN,
        hidden_dim          = b["hidden_dim"],
        n_rnn_layers        = b["n_rnn_layers"],
        hidden_fc_sizes     = fc_map[b["hidden_fc_sizes"]],
        dropout             = b["dropout"],
        batch_size          = b["batch_size"],
        n_epochs            = 100,
        optimizer_kwargs    = {"lr": b["lr"], "weight_decay": b["weight_decay"]},
        random_state        = RANDOM_STATE,
                                add_encoders        =                 {"cyclic": {
                    "past": ["month", "week", "dayofyear", "dayofweek", "day"]
                           }},
        pl_trainer_kwargs   = _nn_trainer_kwargs(),    # no trial -> no pruning callback
    )
    objective_kind = variant.split("_", 1)[1]          # "poisson" / "tweedie"
    return lambda: build_lstm_count(objective_kind, params, tweedie_power=tw_p)


for variant, best_params in best_params_by_variant.items():
    name = f"{variant}_tuned"
    if name in long_by_model:
        print(f"skip {name} — already cached in long_by_model")
        continue

    print(f"\n=== Re-running val CV with best {variant} params ===")

    # --- ADD CONDITIONAL COVARIATE ROUTING ---
    if variant in NN_VARIANTS:
        builder   = _build_lstm_from_best(variant, best_params)
        is_neural = True
        chosen_past_covs = full_raw_past_covs_LSTM
    else:
        builder   = lambda p=best_params, v=variant: build_gbm_from_params(v, p)
        is_neural = False
        chosen_past_covs = full_past_covs

    fold_preds = run_expanding_cv(
        builder,
        target_list    = target_for_cv,
        start_frac     = CV_START_VAL,
        horizon        = OUTPUT_CHUNK_LEN,
        stride         = CV_STRIDE,
        retrain_stride = OUTPUT_CHUNK_LEN,
        past_covs      = chosen_past_covs,
        future_covs    = full_fut_covs,
        is_neural      = is_neural,
    )

    long_df = collect_predictions_long(target_for_cv, fold_preds, region_names)

    long_by_model[name]       = long_df
    fold_preds_by_model[name] = fold_preds

    with open(CKPT / f"{name}.pkl", "wb") as f:
        pickle.dump((long_df, fold_preds), f)


=== Re-running val CV with best lightgbm_poisson params ===
   retrain 1  (data up to 2024-05-11)
   retrain 2  (data up to 2024-05-18)
   retrain 3  (data up to 2024-05-25)
   retrain 4  (data up to 2024-06-01)
   retrain 5  (data up to 2024-06-08)
   retrain 6  (data up to 2024-06-15)
   retrain 7  (data up to 2024-06-22)
   retrain 8  (data up to 2024-06-29)
   retrain 9  (data up to 2024-07-06)
   retrain 10  (data up to 2024-07-13)
   retrain 11  (data up to 2024-07-20)
   retrain 12  (data up to 2024-07-27)
   79 predictions, 12 retrains complete

=== Re-running val CV with best lightgbm_tweedie params ===
   retrain 1  (data up to 2024-05-11)
   retrain 2  (data up to 2024-05-18)
   retrain 3  (data up to 2024-05-25)
   retrain 4  (data up to 2024-06-01)
   retrain 5  (data up to 2024-06-08)
   retrain 6  (data up to 2024-06-15)
   retrain 7  (data up to 2024-06-22)
   retrain 8  (data up to 2024-06-29)
   retrain 9  (data up to 2024-07-06)
   retrain 10  (data up to 2024-07-13

## 10. Leaderboard - which Regressor wins overall?

In [ ]:
results_by_model = {}
for name, long_df in long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES)
    results_by_model[name] = res

In [ ]:
# Leaderboard — sorted by MASE_mean (scale-free, comparable across regions).
# Skill scores give a direct sanity check:
#   Skill = 1 - metric(model) / metric(naive_weekly_OOS)
#     > 0  model is better than the out-of-sample seasonal naive
#     = 0  same
#     < 0  worse than doing nothing smart
# If `naive_weekly` wasn't run, skill columns stay NaN.
ref_name = "naive_weekly"
ref_g    = results_by_model.get(ref_name, {}).get("global", None)

# _skill imported from src

leaderboard_rows = []
for name, res in results_by_model.items():
    g = res["global"]
    leaderboard_rows.append({
        "model":       name,
        "MAE":         g["MAE"],
        "RMSE":        g["RMSE"],
        "MedAE":       g["MedAE"],
        "ME":          g["ME"],
        "PoissonDev":  g["PoissonDev"],
        "TweedieDev":  g["TweedieDev"],
        "ZeroAcc":     g["ZeroAcc"],
        "MASE_mean":   g["MASE_mean"],
        "MASE_median": g["MASE_median"],
        "RMSSE_mean":  g["RMSSE_mean"],
        "SkillMAE":    _skill(g["MAE"],        ref_g["MAE"])        if ref_g else float("nan"),
        "SkillRMSE":   _skill(g["RMSE"],       ref_g["RMSE"])       if ref_g else float("nan"),
        "SkillMASE":   _skill(g["MASE_mean"],  ref_g["MASE_mean"])  if ref_g else float("nan"),
        "SkillRMSSE":  _skill(g["RMSSE_mean"], ref_g["RMSSE_mean"]) if ref_g else float("nan"),
    })
leaderboard = (
    pd.DataFrame(leaderboard_rows)
      .sort_values("MASE_mean", ascending=True)
      .reset_index(drop=True)
)
leaderboard


,model,MAE,RMSE,MedAE,ME,PoissonDev,TweedieDev,ZeroAcc,MASE_mean,MASE_median,RMSSE_mean,SkillMAE,SkillRMSE,SkillMASE,SkillRMSSE
0,lightgbm_tweedie_tuned,0.447952,1.080047,0.047754,-0.148235,0.797010,1.669398,0.863562,1.608489,1.397293,0.930266,NaN,NaN,NaN,NaN
1,xgboost_tweedie_tuned,0.459466,1.065317,0.068432,-0.112940,0.772446,1.619041,0.871338,1.820954,1.572982,0.928569,NaN,NaN,NaN,NaN
2,xgboost_poisson_tuned,0.459983,1.067066,0.070546,-0.118613,0.776226,1.622132,0.869711,1.876932,1.603526,0.930461,NaN,NaN,NaN,NaN
3,lightgbm_poisson_tuned,0.462950,1.062113,0.074606,-0.104433,0.771610,1.623961,0.862749,1.880619,1.576802,0.928536,NaN,NaN,NaN,NaN
4,catboost_tweedie_tuned,0.457992,1.048389,0.071229,-0.085826,0.761998,1.608772,0.872694,1.884521,1.678643,0.924537,NaN,NaN,NaN,NaN
5,catboost_poisson_tuned,0.461657,1.069009,0.076339,-0.099871,0.779891,1.636177,0.868626,2.093845,1.843182,0.935515,NaN,NaN,NaN,NaN


## 13. Per-region detail for the winning model

In [ ]:
winner = leaderboard.iloc[0]["model"]
print(f"Winning model on MASE_mean: {winner}")
results_by_model[winner]["per_region"]


Winning model on MASE_mean: lightgbm_tweedie_tuned


,MAE,RMSE,MedAE,ME,PoissonDev,TweedieDev,ZeroAcc,n,MASE,RMSSE,region
0,0.130517,0.206677,0.092197,0.062274,0.308201,1.450730,0.962025,553,0.506510,0.367837,odesa
1,0.038097,0.112973,0.024536,0.013463,0.119237,0.849424,0.987342,553,0.558116,0.379246,khmelnytskyi
2,0.012803,0.013967,0.011613,0.012803,0.025607,0.443469,1.000000,553,0.625229,0.084525,luhansk
3,0.145138,0.266809,0.080010,0.003737,0.388261,1.572019,0.922242,553,0.752664,0.515465,mykolaiv
4,0.177254,0.329271,0.101738,0.014371,0.488848,1.814633,0.924051,553,0.858439,0.402074,donetsk
5,0.138692,0.274784,0.068187,-0.013004,0.418953,1.671758,0.918626,553,1.041967,0.566241,kyiv
6,0.032027,0.112931,0.017936,0.007073,0.122422,0.882115,0.987342,553,1.042658,0.644359,cherkasy
7,0.050499,0.191397,0.011699,-0.023832,0.251039,1.264033,0.962025,553,1.138180,0.794593,zhytomyr
8,0.047627,0.249764,0.008703,-0.027706,0.318301,1.617339,0.974684,553,1.162896,1.007693,lviv
9,0.049341,0.162184,0.022131,-0.003437,0.190722,1.063319,0.972875,553,1.314255,0.769963,vinnytsia


In [ ]:
winner = leaderboard.iloc[0]["model"]
print(f"Winning model on MASE_mean: {winner}")
results_by_model[winner]["per_horizon"]


Winning model on MASE_mean: lightgbm_tweedie_tuned


,MAE,RMSE,MedAE,ME,PoissonDev,TweedieDev,ZeroAcc,n,horizon
0,0.443492,1.088461,0.046277,-0.155504,0.813537,1.712668,0.858861,1580,1
1,0.440440,1.079501,0.046136,-0.161644,0.791775,1.666260,0.865190,1580,2
2,0.439755,1.075478,0.048532,-0.156339,0.790677,1.670038,0.866456,1580,3
3,0.446603,1.079655,0.047872,-0.148592,0.797366,1.658986,0.863291,1580,4
4,0.447833,1.061158,0.047568,-0.129186,0.779858,1.641223,0.863291,1580,5
5,0.457589,1.083242,0.051765,-0.142659,0.793483,1.649945,0.866456,1580,6
6,0.459956,1.092550,0.045535,-0.143719,0.812376,1.686665,0.861392,1580,7


In [ ]:
target_full = [tr.append(vl).append(te)
               for tr, vl, te in zip(train_target, val_target, test_target)]
TEST_START_FRAC = TRAIN_VAL_END

test_long_by_model       = {}
test_fold_preds_by_model = {}

for variant, best_params in best_params_by_variant.items():
    name = f"{variant}_tuned"
    print(f"\n=== Test-set CV: {name} ===")

    if variant in NN_VARIANTS:
        builder   = _build_lstm_from_best(variant, best_params)
        is_neural = True
    else:
        builder   = lambda p=best_params, v=variant: build_gbm_from_params(v, p)
        is_neural = False

    fold_preds = run_final_test(
        builder,
        target_list    = target_full,
        start_frac     = TEST_START_FRAC,
        predict_stride = 1,
        retrain_stride = OUTPUT_CHUNK_LEN,
        horizon        = OUTPUT_CHUNK_LEN,
        past_covs      = full_past_covs,
        future_covs    = full_fut_covs,
        is_neural      = is_neural,
    )
    long_df = collect_predictions_long(target_full, fold_preds, region_names)
    test_long_by_model[name]       = long_df
    test_fold_preds_by_model[name] = fold_preds

test_results_by_model = {n: evaluate_long(df, MAE_SCALES, RMSE_SCALES) for n, df in test_long_by_model.items()}

test_rows = []
for name, res in test_results_by_model.items():
    g = res["global"]
    test_rows.append({
        "model":      name,
        "MAE":        g["MAE"],
        "RMSE":       g["RMSE"],
        "MedAE":      g["MedAE"],
        "ME":         g["ME"],
        "PoissonDev": g["PoissonDev"],
        "TweedieDev": g["TweedieDev"],
        "ZeroAcc":    g["ZeroAcc"],
        "MASE_mean":  g["MASE_mean"],
        "RMSSE_mean": g["RMSSE_mean"],
    })

test_leaderboard = (
    pd.DataFrame(test_rows)
      .sort_values("MASE_mean", ascending=True)
      .reset_index(drop=True)
)
print(test_leaderboard)



=== Test-set CV: lightgbm_poisson_tuned ===
   retrain 1  (data up to 2024-08-05)
   retrain 2  (data up to 2024-08-12)
   retrain 3  (data up to 2024-08-19)
   retrain 4  (data up to 2024-08-26)
   retrain 5  (data up to 2024-09-02)
   retrain 6  (data up to 2024-09-09)
   retrain 7  (data up to 2024-09-16)
   retrain 8  (data up to 2024-09-23)
   retrain 9  (data up to 2024-09-30)
   retrain 10  (data up to 2024-10-07)
   retrain 11  (data up to 2024-10-14)
   retrain 12  (data up to 2024-10-21)
   retrain 13  (data up to 2024-10-28)
   retrain 14  (data up to 2024-11-04)
   retrain 15  (data up to 2024-11-11)
   retrain 16  (data up to 2024-11-18)
   retrain 17  (data up to 2024-11-25)
   retrain 18  (data up to 2024-12-02)
   retrain 19  (data up to 2024-12-09)
   retrain 20  (data up to 2024-12-16)
   retrain 21  (data up to 2024-12-23)
   retrain 22  (data up to 2024-12-30)
   retrain 23  (data up to 2025-01-06)
   retrain 24  (data up to 2025-01-13)
   164 daily predictions, 24

In [ ]:
target_full = [tr.append(vl).append(te)
               for tr, vl, te in zip(train_target, val_target, test_target)]
TEST_START_FRAC = TRAIN_VAL_END

test_long_by_model       = {}
test_fold_preds_by_model = {}

for variant, best_params in best_params_by_variant.items():
    name = f"{variant}_tuned"
    print(f"\n=== Test-set CV: {name} ===")

    if variant in NN_VARIANTS:
        builder   = _build_lstm_from_best(variant, best_params)
        is_neural = True
    else:
        builder   = lambda p=best_params, v=variant: build_gbm_from_params(v, p)
        is_neural = False

    fold_preds = run_final_test(
        builder,
        target_list    = target_full,
        start_frac     = TEST_START_FRAC,
        predict_stride = 1,
        retrain_stride = OUTPUT_CHUNK_LEN,
        horizon        = OUTPUT_CHUNK_LEN,
        past_covs      = full_past_covs,
        future_covs    = full_fut_covs,
        is_neural      = is_neural,
    )
    long_df = collect_predictions_long(target_full, fold_preds, region_names)
    test_long_by_model[name]       = long_df
    test_fold_preds_by_model[name] = fold_preds

test_results_by_model = {n: evaluate_long(df, MAE_SCALES, RMSE_SCALES) for n, df in test_long_by_model.items()}

test_rows = []
for name, res in test_results_by_model.items():
    g = res["global"]
    test_rows.append({
        "model":      name,
        "MAE":        g["MAE"],
        "RMSE":       g["RMSE"],
        "MedAE":      g["MedAE"],
        "ME":         g["ME"],
        "PoissonDev": g["PoissonDev"],
        "TweedieDev": g["TweedieDev"],
        "ZeroAcc":    g["ZeroAcc"],
        "MASE_mean":  g["MASE_mean"],
        "RMSSE_mean": g["RMSSE_mean"],
    })

test_leaderboard = (
    pd.DataFrame(test_rows)
      .sort_values("MASE_mean", ascending=True)
      .reset_index(drop=True)
)
print(test_leaderboard)



=== Test-set CV: lightgbm_poisson_tuned ===
   retrain 1  (data up to 2024-08-05)
   retrain 2  (data up to 2024-08-12)
   retrain 3  (data up to 2024-08-19)
   retrain 4  (data up to 2024-08-26)
   retrain 5  (data up to 2024-09-02)
   retrain 6  (data up to 2024-09-09)
   retrain 7  (data up to 2024-09-16)
   retrain 8  (data up to 2024-09-23)
   retrain 9  (data up to 2024-09-30)
   retrain 10  (data up to 2024-10-07)
   retrain 11  (data up to 2024-10-14)
   retrain 12  (data up to 2024-10-21)
   retrain 13  (data up to 2024-10-28)
   retrain 14  (data up to 2024-11-04)
   retrain 15  (data up to 2024-11-11)
   retrain 16  (data up to 2024-11-18)
   retrain 17  (data up to 2024-11-25)
   retrain 18  (data up to 2024-12-02)
   retrain 19  (data up to 2024-12-09)
   retrain 20  (data up to 2024-12-16)
   retrain 21  (data up to 2024-12-23)
   retrain 22  (data up to 2024-12-30)
   retrain 23  (data up to 2025-01-06)
   retrain 24  (data up to 2025-01-13)
   164 daily predictions, 24

## Per-activity-level and per-region model comparison

Each tuned variant is re-run in two additional training paradigms:

- **Activity-level model**: one model per conflict-intensity group (low / medium / high), trained only on the homogeneous subset of regions it owns.
- **Local model**: one model per region (20 independent models), each trained exclusively on its own series and covariates.

Both use `predict_stride=1` / `retrain_stride=OUTPUT_CHUNK_LEN` (same decoupled stride as the test evaluation above).
Results are compared against the global model in a leaderboard at the end of this section.

In [ ]:
activity_cv_long_by_model = {}
local_cv_long_by_model    = {}

for variant, best_params in best_params_by_variant.items():
    name = f"{variant}_tuned"
    if variant in NN_VARIANTS:
        builder   = _build_lstm_from_best(variant, best_params)
        is_neural = True
    else:
        builder   = lambda p=best_params, v=variant: build_gbm_from_params(v, p)
        is_neural = False

    print(f"\n=== Activity-level CV: {name} ===")
    fp = run_expanding_cv_per_activity(
        builder, target_for_cv, region_names, regions_activity, CV_START_VAL,
        horizon=OUTPUT_CHUNK_LEN, predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN,
        past_covs=full_past_covs, future_covs=full_fut_covs, is_neural=is_neural,
    )
    activity_cv_long_by_model[name] = collect_predictions_long(target_for_cv, fp, region_names)

    print(f"\n=== Local (per-region) CV: {name} ===")
    fp = run_expanding_cv_per_region(
        builder, target_for_cv, CV_START_VAL,
        horizon=OUTPUT_CHUNK_LEN, predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN,
        past_covs=full_past_covs, future_covs=full_fut_covs, is_neural=is_neural,
    )
    local_cv_long_by_model[name] = collect_predictions_long(target_for_cv, fp, region_names)



=== Activity-level CV: lightgbm_poisson_tuned ===

--- Activity level 1 (10 regions) ---
   retrain 1  (data up to 2024-05-11)
   retrain 2  (data up to 2024-05-18)
   retrain 3  (data up to 2024-05-25)
   retrain 4  (data up to 2024-06-01)
   retrain 5  (data up to 2024-06-08)
   retrain 6  (data up to 2024-06-15)
   retrain 7  (data up to 2024-06-22)
   retrain 8  (data up to 2024-06-29)
   retrain 9  (data up to 2024-07-06)
   retrain 10  (data up to 2024-07-13)
   retrain 11  (data up to 2024-07-20)
   retrain 12  (data up to 2024-07-27)
   79 daily predictions, 12 retrains complete

--- Activity level 2 (4 regions) ---
   retrain 1  (data up to 2024-05-11)
   retrain 2  (data up to 2024-05-18)
   retrain 3  (data up to 2024-05-25)
   retrain 4  (data up to 2024-06-01)
   retrain 5  (data up to 2024-06-08)
   retrain 6  (data up to 2024-06-15)
   retrain 7  (data up to 2024-06-22)
   retrain 8  (data up to 2024-06-29)
   retrain 9  (data up to 2024-07-06)
   retrain 10  (data up t

In [ ]:
activity_test_long_by_model = {}
local_test_long_by_model    = {}

for variant, best_params in best_params_by_variant.items():
    name = f"{variant}_tuned"
    if variant in NN_VARIANTS:
        builder   = _build_lstm_from_best(variant, best_params)
        is_neural = True
    else:
        builder   = lambda p=best_params, v=variant: build_gbm_from_params(v, p)
        is_neural = False

    print(f"\n=== Activity-level test: {name} ===")
    fp = run_expanding_cv_per_activity(
        builder, target_full, region_names, regions_activity, TEST_START_FRAC,
        horizon=OUTPUT_CHUNK_LEN, predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN,
        past_covs=full_past_covs, future_covs=full_fut_covs, is_neural=is_neural,
    )
    activity_test_long_by_model[name] = collect_predictions_long(target_full, fp, region_names)

    print(f"\n=== Local (per-region) test: {name} ===")
    fp = run_expanding_cv_per_region(
        builder, target_full, TEST_START_FRAC,
        horizon=OUTPUT_CHUNK_LEN, predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN,
        past_covs=full_past_covs, future_covs=full_fut_covs, is_neural=is_neural,
    )
    local_test_long_by_model[name] = collect_predictions_long(target_full, fp, region_names)



=== Activity-level test: lightgbm_poisson_tuned ===

--- Activity level 1 (10 regions) ---
   retrain 1  (data up to 2024-08-05)
   retrain 2  (data up to 2024-08-12)
   retrain 3  (data up to 2024-08-19)
   retrain 4  (data up to 2024-08-26)
   retrain 5  (data up to 2024-09-02)
   retrain 6  (data up to 2024-09-09)
   retrain 7  (data up to 2024-09-16)
   retrain 8  (data up to 2024-09-23)
   retrain 9  (data up to 2024-09-30)
   retrain 10  (data up to 2024-10-07)
   retrain 11  (data up to 2024-10-14)
   retrain 12  (data up to 2024-10-21)
   retrain 13  (data up to 2024-10-28)
   retrain 14  (data up to 2024-11-04)
   retrain 15  (data up to 2024-11-11)
   retrain 16  (data up to 2024-11-18)
   retrain 17  (data up to 2024-11-25)
   retrain 18  (data up to 2024-12-02)
   retrain 19  (data up to 2024-12-09)
   retrain 20  (data up to 2024-12-16)
   retrain 21  (data up to 2024-12-23)
   retrain 22  (data up to 2024-12-30)
   retrain 23  (data up to 2025-01-06)
   retrain 24  (data

In [ ]:
import pandas as pd

# Build per-model leaderboard rows (CV validation set)
lb_rows = []
for name, long_df in long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_rows.append({"model": name, "paradigm": "global",   **res["global"]})

for name, long_df in activity_cv_long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_rows.append({"model": name, "paradigm": "activity", **res["global"]})

for name, long_df in local_cv_long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_rows.append({"model": name, "paradigm": "local",    **res["global"]})

cv_leaderboard = pd.DataFrame(lb_rows).sort_values("MASE_mean").reset_index(drop=True)
print("=== CV leaderboard (global vs activity-level vs local) ===")
print(cv_leaderboard)

# Test-set leaderboard
lb_test_rows = []
for name, long_df in test_long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_test_rows.append({"model": name, "paradigm": "global",   **res["global"]})

for name, long_df in activity_test_long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_test_rows.append({"model": name, "paradigm": "activity", **res["global"]})

for name, long_df in local_test_long_by_model.items():
    res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
    lb_test_rows.append({"model": name, "paradigm": "local",    **res["global"]})

test_leaderboard = pd.DataFrame(lb_test_rows).sort_values("MASE_mean").reset_index(drop=True)
print("\n=== Test leaderboard (global vs activity-level vs local) ===")
print(test_leaderboard)


=== CV leaderboard (global vs activity-level vs local) ===
                     model  paradigm       MAE      RMSE     MedAE        ME  PoissonDev  TweedieDev   ZeroAcc      n  MASE_mean  MASE_median  RMSSE_mean
0    xgboost_tweedie_tuned     local  0.431689  1.125911  0.012445 -0.250690    1.031423    2.717945  0.868716  11060   1.344318     1.001527    0.955777
1   lightgbm_tweedie_tuned  activity  0.437291  1.096704  0.018855 -0.172044    0.851512    2.343498  0.870886  11060   1.349459     0.961328    0.930432
2   catboost_tweedie_tuned     local  0.443507  1.114370  0.040582 -0.180339    0.855258    1.845216  0.870976  11060   1.463676     1.135469    0.937950
3   lightgbm_tweedie_tuned     local  0.443751  1.054490  0.048178 -0.114818    0.830869    2.589212  0.870796  11060   1.469756     1.084313    0.932583
4   catboost_poisson_tuned     local  0.462935  1.213316  0.029466 -0.227319    1.004112    2.126275  0.865823  11060   1.503611     1.236417    0.979628
5    xgboost_twee

In [ ]:
# ── Persist all results ───────────────────────────────────────────────────────
import json
from pathlib import Path

RESULTS_DIR = Path("results/gbdt")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _safe(name):
    return name.replace("/", "_").replace(" ", "_")

def _save_group(long_dict, split, paradigm):
    rows = []
    for name, long_df in long_dict.items():
        key = f"{split}_{paradigm}_{_safe(name)}"
        res = evaluate_long(long_df, MAE_SCALES, RMSE_SCALES, regions_activity)
        long_df.to_parquet(RESULTS_DIR / f"predictions_long_{key}.parquet", index=False)
        res["per_region"].to_csv(         RESULTS_DIR / f"per_region_{key}.csv",         index=False)
        res["per_horizon"].to_csv(        RESULTS_DIR / f"per_horizon_{key}.csv",        index=False)
        res["per_region_horizon"].to_csv( RESULTS_DIR / f"per_region_horizon_{key}.csv", index=False)
        with open(RESULTS_DIR / f"global_{key}.json", "w") as fh:
            json.dump(res["global"], fh, indent=2, default=float)
        for view in ("per_activity_level", "per_activity_horizon"):
            if view in res:
                res[view].to_csv(RESULTS_DIR / f"{view}_{key}.csv", index=False)
        rows.append({"split": split, "paradigm": paradigm, "model": name, **res["global"]})
        print(f"  saved: {split}/{paradigm}/{name}")
    return rows

lb_rows = []
lb_rows += _save_group(long_by_model,                                      "cv",   "global")
lb_rows += _save_group(activity_cv_long_by_model,                          "cv",   "activity")
lb_rows += _save_group(globals().get("local_cv_long_by_model",   {}),      "cv",   "local")
lb_rows += _save_group(test_long_by_model,                                 "test", "global")
lb_rows += _save_group(activity_test_long_by_model,                        "test", "activity")
lb_rows += _save_group(globals().get("local_test_long_by_model", {}),      "test", "local")

pd.DataFrame(lb_rows).sort_values(["split", "MASE_mean"]).reset_index(drop=True).to_csv(
    RESULTS_DIR / "leaderboard.csv", index=False
)
print(f"All results in: {RESULTS_DIR.resolve()}")
print(sorted(p.name for p in RESULTS_DIR.iterdir()))


  saved: cv/global/lightgbm_poisson_tuned
  saved: cv/global/lightgbm_tweedie_tuned
  saved: cv/global/xgboost_poisson_tuned
  saved: cv/global/xgboost_tweedie_tuned
  saved: cv/global/catboost_poisson_tuned
  saved: cv/global/catboost_tweedie_tuned
  saved: cv/activity/lightgbm_poisson_tuned
  saved: cv/activity/lightgbm_tweedie_tuned
  saved: cv/activity/xgboost_poisson_tuned
  saved: cv/activity/xgboost_tweedie_tuned
  saved: cv/activity/catboost_poisson_tuned
  saved: cv/activity/catboost_tweedie_tuned
  saved: cv/local/lightgbm_poisson_tuned
  saved: cv/local/lightgbm_tweedie_tuned
  saved: cv/local/xgboost_poisson_tuned
  saved: cv/local/xgboost_tweedie_tuned
  saved: cv/local/catboost_poisson_tuned
  saved: cv/local/catboost_tweedie_tuned
  saved: test/global/lightgbm_poisson_tuned
  saved: test/global/lightgbm_tweedie_tuned
  saved: test/global/xgboost_poisson_tuned
  saved: test/global/xgboost_tweedie_tuned
  saved: test/global/catboost_poisson_tuned
  saved: test/global/catbo

dict_items([('lightgbm_poisson', {'num_leaves': 15, 'max_depth': 8, 'min_child_samples': 168, 'learning_rate': 0.012208717313899137, 'n_estimators': 700, 'subsample': 0.7968651489958548, 'colsample_bytree': 0.8263027234854419, 'reg_alpha': 0.1036638868013055, 'reg_lambda': 0.6417244132141439}), ('lightgbm_tweedie', {'num_leaves': 17, 'max_depth': 10, 'min_child_samples': 168, 'learning_rate': 0.018891200276189388, 'n_estimators': 300, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'reg_alpha': 0.12561043700013558, 'reg_lambda': 0.05342937261279776, 'tweedie_variance_power': 1.3329833121584336}), ('xgboost_poisson', {'max_depth': 3, 'min_child_weight': 9, 'learning_rate': 0.03654613902394278, 'n_estimators': 700, 'subsample': 0.6453122838387236, 'colsample_bytree': 0.6710559300757173, 'reg_alpha': 0.002432579149558049, 'reg_lambda': 0.0036107454705973092}), ('xgboost_tweedie', {'max_depth': 3, 'min_child_weight': 2, 'learning_rate': 0.01731752272762223, 'n_esti

In [ ]:
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance

def get_gbm_importances(fitted_model, model_name_prefix, series, past_covs, fut_covs):
    """
    Extracts both native feature importance (gain) and sklearn permutation importance
    from Darts GBMs per horizon.
    """
    underlying = fitted_model.model
    estimators = getattr(underlying, "estimators_", [underlying])
    features = fitted_model.lagged_feature_names

    df_imps = {"Feature": features}

    # 1. Native Booster Importances (Gain)
    for h, est in enumerate(estimators, start=1):
        if "lightgbm" in model_name_prefix:
            imp = est.booster_.feature_importance(importance_type='gain')
        elif "catboost" in model_name_prefix:
            imp = est.get_feature_importance()
        else:
            imp = est.feature_importances_
        df_imps[f"h{h}_gain"] = imp

    # 2. Sklearn Permutation Importances
    # Apply encoders if present to get the full feature set
    if hasattr(fitted_model, 'encoders') and fitted_model.encoders.encoding_available:
        past_covs_enc, fut_covs_enc = fitted_model.encoders.encode_inference(
            n=fitted_model.output_chunk_length,
            target=series,
            past_covariates=past_covs,
            future_covariates=fut_covs
        )
        series_enc = series
    else:
        series_enc, past_covs_enc, fut_covs_enc = series, past_covs, fut_covs

    # Extract the tabular data created by Darts for the underlying sklearn-like model
    lagged_data = fitted_model._create_lagged_data(
        series=series_enc,
        past_covariates=past_covs_enc,
        future_covariates=fut_covs_enc,
        max_samples_per_ts=None,
        sample_weight=None,
    )
    X, y_target = lagged_data[0], lagged_data[1]

    for h, est in enumerate(estimators, start=1):
        y_h = y_target[:, h - 1]  # target for horizon h
        r = permutation_importance(est, X, y_h, n_repeats=5, random_state=42, n_jobs=-1)
        df_imps[f"h{h}_perm"] = r.importances_mean

    df = pd.DataFrame(df_imps)

    # Aggregate across horizons
    gain_cols = [c for c in df.columns if c.endswith("_gain")]
    perm_cols = [c for c in df.columns if c.endswith("_perm")]

    df["agg_gain"] = df[gain_cols].mean(axis=1)
    df["agg_perm"] = df[perm_cols].mean(axis=1)

    # Sort by permutation importance primarily
    return df.sort_values("agg_perm", ascending=False).reset_index(drop=True)

final_importances = {}

# --- 1. Global: lightgbm_poisson_tuned ---
print("Fitting global lightgbm_poisson...")
builder_lgb_p = lambda: build_gbm_from_params('lightgbm_poisson', best_params_by_variant['lightgbm_poisson'])
builder_cb_t = lambda: build_gbm_from_params('catboost_tweedie', best_params_by_variant['catboost_tweedie'])

global_lgb_p = builder_lgb_p()
# global_lgb_p.fit(series=target_full, past_covariates=full_past_covs, future_covariates=full_fut_covs)

# final_importances["global_lightgbm_poisson"] = get_gbm_importances(
#     global_lgb_p, "lightgbm", series=target_full, past_covs=full_past_covs, fut_covs=full_fut_covs
# )
global_cb_t = builder_cb_t()
global_cb_t.fit(series=target_full, past_covariates=full_past_covs, future_covariates=full_fut_covs)

final_importances["global_catboost_tweedie"] = get_gbm_importances(
    global_cb_t, "catboost", series=target_full, past_covs=full_past_covs, fut_covs=full_fut_covs
)

# --- 2. Activity-level fits ---
# Group target_full and covariates by activity level
activity_levels = sorted(list(set(regions_activity.values())))
targets_by_activity = {lvl: [] for lvl in activity_levels}
past_covs_by_activity = {lvl: [] for lvl in activity_levels}
fut_covs_by_activity = {lvl: [] for lvl in activity_levels}

for ts, pc, fc, reg in zip(target_full, full_past_covs, full_fut_covs, region_names):
    lvl = regions_activity[reg]
    targets_by_activity[lvl].append(ts)
    past_covs_by_activity[lvl].append(pc)
    fut_covs_by_activity[lvl].append(fc)


for lvl in activity_levels:
    ts_group = targets_by_activity[lvl]
    pc_group = past_covs_by_activity[lvl]
    fc_group = fut_covs_by_activity[lvl]
    if not ts_group:
        continue
    print(f"Fitting activity level {lvl} (N={len(ts_group)} regions)...")

    # LightGBM Poisson (Activity)
    m_lgb = builder_lgb_p()
    m_lgb.fit(series=ts_group, past_covariates=pc_group, future_covariates=fc_group)
    final_importances[f"activity_{lvl}_lightgbm_poisson"] = get_gbm_importances(
        m_lgb, "lightgbm", series=ts_group, past_covs=pc_group, fut_covs=fc_group
    )

    # CatBoost Tweedie (Activity)
    m_cb = builder_cb_t()
    m_cb.fit(series=ts_group, past_covariates=pc_group, future_covariates=fc_group)
    final_importances[f"activity_{lvl}_catboost_tweedie"] = get_gbm_importances(
        m_cb, "catboost", series=ts_group, past_covs=pc_group, fut_covs=fc_group
    )

# Preview the global model importances
display(final_importances["global_catboost_tweedie"].head(10))


Fitting global lightgbm_poisson...
Fitting activity level 1 (N=10 regions)...
Fitting activity level 2 (N=4 regions)...
Fitting activity level 3 (N=6 regions)...


,Feature,h1_gain,h2_gain,h3_gain,h4_gain,h5_gain,h6_gain,h7_gain,h1_perm,h2_perm,h3_perm,h4_perm,h5_perm,h6_perm,h7_perm,agg_gain,agg_perm
0,Activity_Level_statcov_target_act_drone_strike...,28.443643,29.131206,30.721274,30.252816,29.599884,30.253383,30.740478,0.239728,0.222473,0.231669,0.228353,0.237896,0.244952,0.267360,29.877526,0.238918
1,region_statcov_target_act_drone_strike_on_ua,2.654972,2.088113,2.215523,2.429220,1.936533,1.843442,1.949863,0.080943,0.067070,0.071479,0.072833,0.060504,0.063343,0.062312,2.159667,0.068355
2,act_drone_strike_on_ua_target_lag-2,1.614067,4.481524,1.766579,2.160474,1.634058,8.972223,5.138844,0.017138,0.038493,0.013098,0.015958,0.016420,0.178802,0.079170,3.681110,0.051297
3,act_drone_strike_on_ua_target_lag-3,4.197673,1.464120,2.051015,1.326452,8.395791,5.011773,1.058560,0.039302,0.012791,0.017334,0.012264,0.175592,0.082901,0.006601,3.357912,0.049541
4,act_drone_strike_on_ua_target_lag-5,1.889677,1.387067,8.380185,4.904239,1.361288,2.405836,3.369114,0.015770,0.009368,0.161115,0.082661,0.008180,0.025764,0.032384,3.385343,0.047892
5,act_drone_strike_on_ua_target_lag-4,1.569213,1.949219,1.262669,8.439265,4.535664,1.193133,2.398085,0.014732,0.016358,0.011847,0.165378,0.079960,0.007243,0.024801,3.049607,0.045760
6,act_drone_strike_on_ua_target_lag-7,7.412571,4.578180,1.276421,2.276209,3.011866,1.622105,0.938107,0.154479,0.081772,0.008856,0.024472,0.029090,0.012580,0.006122,3.016494,0.045339
7,act_drone_strike_on_ua_target_lag-6,1.618474,7.880180,4.686620,1.656290,2.247847,3.446069,1.678936,0.016005,0.150720,0.077421,0.007756,0.023830,0.029535,0.010814,3.316345,0.045154
8,act_drone_strike_on_ua_target_lag-1,4.514360,1.819501,4.637466,1.970688,1.982649,1.559245,8.923361,0.028200,0.014813,0.041892,0.015222,0.015562,0.016739,0.169965,3.629610,0.043199
9,ewm_ewma14_act_drone_infra_ua_residential_inte...,3.339035,3.357921,3.099526,3.562729,3.537522,2.922088,3.149848,0.031156,0.025175,0.023615,0.028343,0.020718,0.022563,0.034694,3.281238,0.026609


In [1]:
for model_name, imp_df in final_importances.items():
    imp_df.to_csv(RESULTS_DIR / f"importance_{model_name}.csv", index=False)

# Or, to save them all in one combined file with a 'model' column:
imp_all = pd.concat(
    [df.assign(model=name) for name, df in final_importances.items()],
    ignore_index=True
)
imp_all.to_csv(RESULTS_DIR / "importance_all.csv", index=False)
print("Saved feature importances to CSV.")

NameError: name 'final_importances' is not defined

In [ ]:
imp_all

,Feature,h1_gain,h2_gain,h3_gain,h4_gain,h5_gain,h6_gain,h7_gain,h1_perm,h2_perm,h3_perm,h4_perm,h5_perm,h6_perm,h7_perm,agg_gain,agg_perm,model
0,Activity_Level_statcov_target_act_drone_strike...,210793.065790,211876.120438,213634.364525,219644.663978,212520.888852,213548.021427,216578.901231,0.453110,0.425324,0.448254,0.442220,0.444198,0.447649,0.463520,214085.146606,0.446325,global_lightgbm_poisson
1,region_statcov_target_act_drone_strike_on_ua,41483.725222,41879.460060,41788.533530,40931.909833,37392.754097,39332.116455,39006.378917,0.248434,0.240392,0.236148,0.218989,0.208579,0.216851,0.212402,40259.268302,0.225971,global_lightgbm_poisson
2,rolling_rmean28_28_act_total_daily_strike_even...,7573.029434,16094.877174,28006.619752,36963.652073,22739.056760,7254.845004,4596.643768,0.025153,0.052312,0.099277,0.137158,0.083459,0.006871,0.004908,17604.103423,0.058448,global_lightgbm_poisson
3,act_drone_strike_on_ua_target_lag-6,3753.572522,281392.908412,80365.639524,7965.260888,7270.683859,33507.506333,4148.334352,0.002977,0.208255,0.067889,0.003444,0.015086,0.015506,0.001139,59771.986556,0.044899,global_lightgbm_poisson
4,act_drone_strike_on_ua_target_lag-5,7035.511613,2388.821548,263471.788999,74188.726727,6944.896626,4913.568590,27734.591531,0.001906,0.002662,0.199127,0.063143,0.004027,0.014853,0.013630,55239.700805,0.042764,global_lightgbm_poisson
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,darts_enc_fc_cyc_month_sin_futcov_lag6,0.000000,0.000000,0.006333,0.007195,0.000000,0.000000,0.000000,0.000000,0.000000,0.000278,0.000113,0.000000,0.000000,0.000000,0.001933,0.000056,activity_3_catboost_tweedie
2033,darts_enc_fc_cyc_week_sin_futcov_lag6,0.041879,0.000959,0.004317,0.001539,0.000000,0.000000,0.014316,0.000143,0.000062,-0.000011,0.000033,0.000000,0.000000,0.000112,0.009001,0.000048,activity_3_catboost_tweedie
2034,darts_enc_fc_cyc_day_cos_futcov_lag0,0.003587,0.003746,0.016649,0.004319,0.000000,0.000000,0.006714,0.000201,0.000023,0.000010,0.000044,0.000000,0.000000,0.000046,0.005002,0.000046,activity_3_catboost_tweedie
2035,darts_enc_fc_cyc_week_sin_futcov_lag0,0.000000,0.000000,0.013608,0.000000,0.000000,0.000000,0.000711,0.000000,0.000000,0.000135,0.000000,0.000000,0.000000,0.000081,0.002045,0.000031,activity_3_catboost_tweedie
